# 02 · Exploratory analysis — first batch

The first collected batch is currently batch 0. All plots use saved data.

## Imports

In [ ]:
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)


## Load batch

In [ ]:
DATA = Path('../data')
GROUP = int(os.environ['DATA_API_GROUP'])
files = sorted(DATA.glob(f'group-{GROUP}-*.json'))
if not files:
    raise FileNotFoundError('Collect a batch in notebook 01 first.')
with files[0].open() as file:
    batch = json.load(file)
df = pd.DataFrame(batch['data'])
print(f"Group {GROUP} | batch {batch['batch_number']} | {len(df):,} rows")


In [ ]:
display(df.head())
df.info()


## Missing values and duplicates

In [ ]:
display(pd.DataFrame({'missing': df.isna().sum(), 'missing_%': df.isna().mean().mul(100).round(2), 'unique': df.nunique()}).sort_values('missing_%', ascending=False))
print('Duplicate rows:', df.duplicated().sum())


## Numerical variables: continuous and discrete


In [ ]:
continuous = ['price', 'acre_lot', 'house_size']
discrete = ['bed', 'bath']
numeric = continuous + discrete
categorical = ['brokered_by', 'status', 'street', 'city', 'state', 'zip_code']
display(df[continuous].describe(percentiles=[.01, .25, .5, .75, .95, .99]).T)
display(df[discrete].describe().T)


In [ ]:
df[numeric].hist(bins=40, figsize=(14, 8))
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['price'].dropna(), bins=50, ax=axes[0])
axes[0].set_title('Price')
sns.histplot(df.loc[df.price.gt(0), 'price'], bins=50, log_scale=(True, False), ax=axes[1])
axes[1].set_title('Positive price (log scale)')
plt.tight_layout()
plt.show()


## Outliers (IQR)

Counts are indicators for investigation, not automatic removal rules.

In [ ]:
q1, q3 = df[numeric].quantile(.25), df[numeric].quantile(.75)
iqr = q3 - q1
outliers = (df[numeric].lt(q1 - 1.5 * iqr) | df[numeric].gt(q3 + 1.5 * iqr))
display(pd.DataFrame({'count': outliers.sum(), 'percent': outliers.mean().mul(100).round(2)}))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=df['price'], ax=axes[0]); axes[0].set_title('Price')
sns.boxplot(x=df['house_size'], ax=axes[1]); axes[1].set_title('House size')
plt.tight_layout()
plt.show()


## Categorical variables — text and numerical codes


In [ ]:
for col in categorical:
    print(f'{col}: {df[col].nunique():,} categories')
    display(df[col].fillna('Missing').value_counts().head(10).rename_axis(col).reset_index(name='count'))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col in zip(axes, ['status', 'state']):
    counts = df[col].fillna('Missing').value_counts().head(10)
    sns.barplot(x=counts.values, y=counts.index, ax=ax)
    ax.set_title(f'Top {col} values')
plt.tight_layout()
plt.show()


In [ ]:
top_states = df['state'].value_counts().head(8).index
plot_data = df[df['state'].isin(top_states) & df['price'].gt(0)]
sns.boxplot(data=plot_data.sample(min(len(plot_data), 10000), random_state=42), x='price', y='state', showfliers=False)
plt.xscale('log')
plt.title('Price by state — sample, without displayed fliers')
plt.show()


## Correlation and relationships

In [ ]:
correlation = df[numeric].corr()
display(correlation)
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Numeric correlation')
plt.show()


In [ ]:
sample = df[numeric].dropna().sample(n=min(1000, len(df[numeric].dropna())), random_state=42)
sns.pairplot(sample, corner=True, plot_kws={'alpha': .3, 's': 10})
plt.show()


## Previous sale date


In [ ]:
sold_date = pd.to_datetime(df['prev_sold_date'], errors='coerce')
print('Missing or invalid dates:', sold_date.isna().sum())
print('Date range:', sold_date.min(), 'to', sold_date.max())
display(sold_date.dt.year.value_counts().sort_index().tail(20).rename_axis('year').reset_index(name='count'))
sold_date.dt.year.hist(bins=40, figsize=(9, 3))
plt.title('Previous sale year')
plt.xlabel('Year')
plt.show()


## Notes

Use this exploration to inspect missing values, extreme prices, and category coverage before choosing model inputs.
